# 06: Conservative Pseudo-Labeling

**Objective:** Expand knowledge safely to the unlabeled segment (~19,900 HCPs).

**Key Actions:**
- Inference over the unlabeled segment using the trained hybrid models.
- Establish a hyper-conservative decision boundary (prob >= 0.98 or prob <= 0.02).
- Assign new synthetic labels to construct the dataset for the next active learning cycle.

In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Assuming 'hybrid_df' and 'trained_models' from notebook 05 are saved/loaded

## 1. Unlabeled Inference

In [ ]:
def infer_unlabeled_segment(df: pd.DataFrame, models: list) -> pd.DataFrame:
    """
    Generates out-of-sample predictions for the fully unlabeled cohorts.
    Evaluates using an ensemble approach (mean of all K-Fold models).
    """
    logger.info("Predicting on the massive unlabeled cohort...")
    if df.empty or not models: return pd.DataFrame()
    
    unlabeled_mask = df['fold'] == -1
    df_unlabeled = df[unlabeled_mask].copy()
    
    if df_unlabeled.empty:
        logger.info("No unlabeled data found.")
        return pd.DataFrame()
        
    features = [c for c in df_unlabeled.columns if c not in ['NUEVO_ID', 'ATSEG', 'fold']]
    X_unlabeled = df_unlabeled[features]
    
    # Ensemble predictions
    fold_preds = np.zeros((len(df_unlabeled), len(models)))
    for i, model in enumerate(models):
        fold_preds[:, i] = model.predict_proba(X_unlabeled)[:, 1]
        
    # Average prediction across folds
    df_unlabeled['ensemble_prob'] = fold_preds.mean(axis=1)
    # Standard deviation to measure uncertainty across folds
    df_unlabeled['ensemble_std'] = fold_preds.std(axis=1)
    
    return df_unlabeled[['NUEVO_ID', 'ensemble_prob', 'ensemble_std']]

# predictions_df = infer_unlabeled_segment(hybrid_df, trained_models)

## 2. Hyper-Conservative Filtering Strategy

In [ ]:
def assign_pseudo_labels(preds_df: pd.DataFrame, upper_thresh: float = 0.98, lower_thresh: float = 0.02) -> pd.DataFrame:
    """
    Extracts only those sequences where the model holds absolute statistical certainty.
    """
    logger.info(f"Applying conservative thresholds: >= {upper_thresh} or <= {lower_thresh}")
    if preds_df.empty: return preds_df
    
    positive_mask = (preds_df['ensemble_prob'] >= upper_thresh)
    negative_mask = (preds_df['ensemble_prob'] <= lower_thresh)
    
    preds_df['pseudo_label'] = np.nan
    preds_df.loc[positive_mask, 'pseudo_label'] = 1
    preds_df.loc[negative_mask, 'pseudo_label'] = 0
    
    pseudo_labeled_df = preds_df[preds_df['pseudo_label'].notna()].copy()
    
    logger.info(f"Discovered {positive_mask.sum()} High-Confidence Positives.")
    logger.info(f"Discovered {negative_mask.sum()} High-Confidence Negatives.")
    logger.info(f"Total synthetic data generated for next cycle: {len(pseudo_labeled_df)}")
    
    return pseudo_labeled_df

# new_synthetic_labels = assign_pseudo_labels(predictions_df)
# new_synthetic_labels.to_parquet('data/labels/pseudo_labels_cycle_1.parquet', index=False)